# W02b.1 — LLM API Basics with LangChain

We'll start with the basics of LLM API calls with LangChain.

The rest of the notebook assumes you've installed the required packages and set up your .env file with the required keys as shown in class.

In [1]:
# import the API keys
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

API key loaded


## 1. Your first call

LangChain provides a small number of core features to get started with AI agents: `chat_models`, `messages`, `tools`, `agents`.

`init_chat_model` is the universal constructor for a chat model: give it a model name, get back a model with the same interface regardless of provider.

### Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
claude = init_chat_model(model="claude-sonnet-4-6")

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [3]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

### Sending an inference request

Use `invoke` to send a request to the model.

In [4]:
question = "In one sentence: What does temperature do in an API call?"

In [6]:
#TODO: send the request and store the response
#TODO: display the response

response = model.invoke(question)
print(response)

content='Temperature determines the diversity and creativity of the generated text by adding random sampling to the token selection process, allowing for less predictable output at higher values.' additional_kwargs={} response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-18T20:06:29.731295Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1309864708, 'load_duration': 3373875, 'prompt_eval_count': 25, 'prompt_eval_duration': 121663000, 'eval_count': 30, 'eval_duration': 1159820000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'} id='lc_run--01a0b620-a741-7f12-b67c-688716089366-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 25, 'output_tokens': 30, 'total_tokens': 55}


`invoke` returned an `AIMessage` object, not a string. It carries the text plus metadata about how it was produced. Look at what else is in there.

In [8]:
response.content

'Temperature determines the diversity and creativity of the generated text by adding random sampling to the token selection process, allowing for less predictable output at higher values.'

In [7]:
from pprint import pprint

print(type(response).__name__)
pprint(response.response_metadata)

AIMessage
{'created_at': '2026-09-18T20:06:29.731295Z',
 'done': True,
 'done_reason': 'stop',
 'eval_count': 30,
 'eval_duration': 1159820000,
 'load_duration': 3373875,
 'logprobs': None,
 'model': 'qwen3.5:4b',
 'model_name': 'qwen3.5:4b',
 'model_provider': 'ollama',
 'prompt_eval_count': 25,
 'prompt_eval_duration': 121663000,
 'total_duration': 1309864708}


## 2. Messages and roles

APIs take a list of messages, each with a role:

| Role | Class | Who it is |
|---|---|---|
| system | `SystemMessage` | You, the developer: persona, rules |
| user | `HumanMessage` | The end user |
| assistant | `AIMessage` | The model's own past turns |

The system prompt is used to steer the assistant persona.

Example: `SystemMessage(content="your message")`

In [9]:
system_prompt = "You are a teaching assistant for an Agentic AI course. Answer in one sentence."
user_question = "What does temperature do in an API call?"

In [10]:
from langchain.messages import SystemMessage, HumanMessage

#TODO: define the list of messages containing system and user prompts.
system_message = SystemMessage(content=system_prompt)
human_message = HumanMessage(content=user_question)

messages = [system_message, human_message]

response = model.invoke(messages)
print(response.content)

In generative AI APIs, the temperature parameter controls the randomness of the model's output to balance between creativity and consistency; lower values (e.g., 0) produce more deterministic and predictable results by limiting word choice based on probability, while higher values increase randomness for more diverse but potentially erratic responses.


In [11]:
response

AIMessage(content="In generative AI APIs, the temperature parameter controls the randomness of the model's output to balance between creativity and consistency; lower values (e.g., 0) produce more deterministic and predictable results by limiting word choice based on probability, while higher values increase randomness for more diverse but potentially erratic responses.", additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-18T20:11:20.435482Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2817691250, 'load_duration': 10258750, 'prompt_eval_count': 43, 'prompt_eval_duration': 317721000, 'eval_count': 61, 'eval_duration': 2460912000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0b625-10f0-7a01-ae20-8142c9e7463c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 61, 'total_tokens': 104})

## 3. The model is stateless

The model keeps no memory between calls. If you want a conversation, you must send the whole history every time. 

Let's see this in action. First turn:

In [12]:
from langchain.messages import AIMessage

conversation = [
    SystemMessage(content="You are a concise tutor."),
    HumanMessage(content="Hello from Boston. Give me a two-line analogy for what an embedding is."),
]

first_response = model.invoke(conversation)


In [13]:
type(first_response)

langchain_core.messages.ai.AIMessage

In [14]:
first_response

AIMessage(content='An embedding maps complex, high-dimensional data into a simple, low-dimensional space like the Cartesian plane (like drawing a map of a city).\nJust as coordinates let you locate any landmark using just numbers, embeddings let computers locate concepts using numerical vectors.', additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-18T20:12:53.62545Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2401976125, 'load_duration': 1514834, 'prompt_eval_count': 39, 'prompt_eval_duration': 295587000, 'eval_count': 51, 'eval_duration': 2085274000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0b626-7e94-7232-a493-76e3f8dc32e9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 51, 'total_tokens': 90})

In [15]:
next_message = HumanMessage(content="Where do I live?")
next_response = model.invoke([next_message])
print(next_response.content)

I don't have access to your personal information, so I cannot tell you where you live. Only you know that. Where is your location?


In [16]:
print(first_response.content)

An embedding maps complex, high-dimensional data into a simple, low-dimensional space like the Cartesian plane (like drawing a map of a city).
Just as coordinates let you locate any landmark using just numbers, embeddings let computers locate concepts using numerical vectors.


To help the model remember previous conversations, we need to append the model's reply to the history:

In [17]:
pprint(conversation)

[SystemMessage(content='You are a concise tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello from Boston. Give me a two-line analogy for what an embedding is.', additional_kwargs={}, response_metadata={})]


In [18]:
conversation.append(first_response)

In [19]:
pprint(conversation)

[SystemMessage(content='You are a concise tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello from Boston. Give me a two-line analogy for what an embedding is.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='An embedding maps complex, high-dimensional data into a simple, low-dimensional space like the Cartesian plane (like drawing a map of a city).\nJust as coordinates let you locate any landmark using just numbers, embeddings let computers locate concepts using numerical vectors.', additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-18T20:12:53.62545Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2401976125, 'load_duration': 1514834, 'prompt_eval_count': 39, 'prompt_eval_duration': 295587000, 'eval_count': 51, 'eval_duration': 2085274000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0b626-7e94-7232-a493-76e3f8dc32e9-0', tool_calls=[], invalid_too

In [20]:
model.invoke([next_message])

AIMessage(content="I don't have access to your personal information, so I can't tell you where you live. Your location is private and not available to me unless you choose to share it with me in this conversation. If you'd like to discuss something related to a specific place (like travel tips for a city or details about a region), feel free to mention the name of the city or area, and I'd be happy to help!", additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-18T20:14:41.856879Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3411127708, 'load_duration': 1893666, 'prompt_eval_count': 17, 'prompt_eval_duration': 128793000, 'eval_count': 87, 'eval_duration': 3253721000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0b628-216b-7b33-9d22-d313a6140631-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 87, 'total_tokens': 104})

We need to append the follow-up to the original list of messages and send everything again:

In [23]:
conversation

[SystemMessage(content='You are a concise tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello from Boston. Give me a two-line analogy for what an embedding is.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='An embedding maps complex, high-dimensional data into a simple, low-dimensional space like the Cartesian plane (like drawing a map of a city).\nJust as coordinates let you locate any landmark using just numbers, embeddings let computers locate concepts using numerical vectors.', additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-18T20:12:53.62545Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2401976125, 'load_duration': 1514834, 'prompt_eval_count': 39, 'prompt_eval_duration': 295587000, 'eval_count': 51, 'eval_duration': 2085274000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0b626-7e94-7232-a493-76e3f8dc32e9-0', tool_calls=[], invalid_too

In [22]:
conversation.append(next_message)

In [24]:
second_message = model.invoke(conversation)
print(second_message.content)

You live in **Boston**.


In [25]:
new_message = HumanMessage("Give me another analogy that's more relatable to someone who loves cooking.")
conversation.append(new_message)

In [26]:
last_response = model.invoke(conversation)

print(last_response.content)

Think of data dimensions like a chef with too many ingredients; they make it hard to see the recipe clearly.
An embedding is like a skilled sous-chef who arranges those ingredients into a simple, tasty dish where the flavors naturally group together.


In [27]:
pprint(conversation)

[SystemMessage(content='You are a concise tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello from Boston. Give me a two-line analogy for what an embedding is.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='An embedding maps complex, high-dimensional data into a simple, low-dimensional space like the Cartesian plane (like drawing a map of a city).\nJust as coordinates let you locate any landmark using just numbers, embeddings let computers locate concepts using numerical vectors.', additional_kwargs={}, response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-09-18T20:12:53.62545Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2401976125, 'load_duration': 1514834, 'prompt_eval_count': 39, 'prompt_eval_duration': 295587000, 'eval_count': 51, 'eval_duration': 2085274000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'}, id='lc_run--01a0b626-7e94-7232-a493-76e3f8dc32e9-0', tool_calls=[], invalid_too

Every extra turn grows the context the model re-reads. This is why long agent sessions get expensive.

## 4. The sampling knobs

As we discussed, the model outputs a probability distribution over the next token. The API knobs decide how we draw from it.

- `temperature` divides the logits before softmax. Low sharpens (more deterministic), high flattens.
- `top_p` keeps only the smallest set of tokens covering probability mass p.
- `max_tokens` is a hard cap on output length.

BUT: reasoning-first models (the `gpt-5` family) fix their sampling internally. For these experiments we'll use a standard chat model without reasoning.

In [ ]:
#sampler = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
sampler = ChatOllama(model="qwen3.5:4b", reasoning=False)
#sampler = init_chat_model(model="gpt-4.1-mini")


Sampling is random, so a single call proves nothing. A small helper to run the same prompt several times:

In [28]:
def sample_n(prompt, n=4, **model_kwargs):
    #m = init_chat_model(model="gpt-4.1-mini", **model_kwargs)
    m = ChatOllama(model="qwen3.5:4b", reasoning=False, **model_kwargs)
    return [m.invoke(prompt).content.strip() for _ in range(n)]

prompt = "Invent a name for Northeastern University's mascot, husky. Reply with the name only."


In [29]:
print("temperature = 0.0  (near-greedy)")

for out in sample_n(prompt, temperature=0.0):
    print(" ", out)

temperature = 0.0  (near-greedy)
  The Husk
  The Husk
  The Husk
  The Husk


In [30]:
print("temperature = 1.4  (flattened distribution)")
for out in sample_n(prompt, temperature=1.4):
    print(" ", out)

temperature = 1.4  (flattened distribution)
  Dawgie
  The Husk
  Huskie
  The Husker


Near-identical names at 0.0, real variety at 1.4. Two caveats: even at temperature 0 the answers can occasionally differ, and high temperature is not more intelligence, only a flatter distribution.

In [31]:
print("top_p = 0.1  (only the most probable tokens survive)")
for out in sample_n(prompt, top_p=0.1):
    print(" ", out)

top_p = 0.1  (only the most probable tokens survive)
  The Husk
  The Husk
  The Husk
  The Husk


### Choosing settings:

| Task | Setting |
|---|---|
| Tool calls, structured output | temperature 0 to 0.3 |
| Factual Q&A, extraction | low 0.3 to 0.5 |
| Brainstorming, drafts | 0.8 to 1.2, several samples |
| General chat | defaults; tune one knob, not both |

**Rule of thumb for agents: agents loop, and a small weirdness rate per step compounds across twenty steps. Keep agent temperature low.**

### max_tokens
It's also possible to limit the number of tokens to be generated. Use `max_tokens` as a cost ceiling. 

In [32]:
capped = ChatOllama(model="qwen3.5:4b", reasoning=False, max_tokens = 50)
#capped = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", max_tokens=50)
r = capped.invoke("Explain how backpropagation works.")


In [33]:
pprint(r.content)
pprint("\nstop reason:", r.response_metadata.get("finish_reason") or r.response_metadata.get("stop_reason"))

('**Backpropagation** is the core algorithm used in training neural networks '
 'to adjust their internal parameters (weights and biases) so that their '
 'predictions become more accurate. It is essentially a specialized '
 'application of the **chain rule from calculus**, used to efficiently '
 'calculate the gradient of the loss function with respect to every parameter '
 'in the network.\n'
 '\n'
 'Here is a step-by-step breakdown of how it works:\n'
 '\n'
 '### 1. The Goal: Minimizing Loss\n'
 'In any training process, we start with an input ($x$) and try to produce a '
 'prediction ($\\hat{y}$). We compare this prediction to the actual target '
 'value ($y$) using a **Loss Function** (e.g., Mean Squared Error or '
 'Cross-Entropy).\n'
 '$$ \\text{Loss} = f(\\hat{y}, y) $$\n'
 'Our goal is to make $\\text{Loss}$ as small as possible by tweaking the '
 'weights in the network.\n'
 '\n'
 '### 2. The Forward Pass\n'
 'Before backpropagation can happen, a **Forward Pass** must occur:\

The text stops mid-sentence and the stop reason says `length`, not `stop`. 

## 5. Streaming

Generation is one token per forward pass. Streaming exposes that rhythm instead of making you wait for the full reply.

Use `model.stream()` to stream each chunk as you receive them. Will need to iterate through the 
- model.stream("your query") will return the response as a sequence

In [35]:
#TODO: iterate through the stream
#TODO: display the chunks
for chunk in model.stream("Write a poem for Northeastern University"):
    print(chunk.content, end="", flush=True)

**The Red Line Rising**

Where the Charles flows with ancient, misty breath,
And brick and glass both meet the summer sky,
A crimson tide begins to take its path,
Beneath the watch of autumn's golden eye.

From Cambridge green, the journey starts anew,
Across the tracks where history has run,
The Red Line hums a rhythm, steady, true,
Connecting stories past and all yet done.

Here walls of ivy cloak the stately halls,
Where professors carve their knowledge into stone,
A tapestry of minds that once and always falls,
To lift the human spirit, make it whole alone.

The library stacks rise like a digital cloud,
Among the trees that sway on Brookline's edge,
A place where ideas find a grounded sound,
And youth takes flight to claim what destiny decrees.

With engineering that seeks the boundless deep,
And liberal arts that ask the soul to grow,
The campus holds the promise of the keep,
That every student here will find their way to know.

So let the leaves turn orange, gold, and brown,
As f